# SQ2 — Amanda: personalisation awareness and perceived control
Run all cells. Place Responses.xlsx beside this file or set BS_DATA_PATH. The survey is not stored in GitHub.


### Import the analysis libraries

Loads pandas for tables, NumPy for numerical operations, Matplotlib for charts, SciPy for statistical functions, and charset-normalizer for encoding utilities. Some imports are retained from Fred’s notebook even though later cells do not use them.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import scipy.stats as stats

### Load the survey
Use BS_DATA_PATH for a local file, or the shared Responses.xlsx in Google Drive. The raw file is never changed.


In [ ]:
import os
from pathlib import Path
if 'google.colab' in __import__('sys').modules:
    from google.colab import drive
    drive.mount('/content/drive')


In [ ]:
candidates = [Path(os.environ.get('BS_DATA_PATH', 'Responses.xlsx')),
              Path('/content/drive/MyDrive/Colab Notebooks/Responses.xlsx'),
              Path.home() / 'Downloads' / 'Responses (3).xlsx']
file_path = next((p for p in candidates if p.is_file()), None)
if file_path is None:
    raise FileNotFoundError('Place Responses.xlsx beside this notebook or set BS_DATA_PATH.')
df = pd.read_excel(file_path)
for col in df.select_dtypes(include=['object', 'string']):
    df[col] = df[col].map(lambda v: v.strip() if isinstance(v, str) else v)
print('Loaded survey:', df.shape)


# SQ2 — Content personalisation and perceived control (Amanda)
**Updated group question:** How does content personalization influence users' perceived control over their social media feeds across different age groups?

**Operational question:** Is awareness of personal-data-based feed shaping associated with perceived control, after accounting for age group? Does this association vary by age group?

The survey has no direct item measuring the amount of personalisation. The predictor is the transparency/awareness statement, “I am informed that my feed is shaped by personal information like my age, gender or location”. This is a limited proxy for awareness of personalisation, not actual exposure to personalisation. The outcome is “I feel I am in control of my feed”. Both use five ordered agreement categories. Do not interpret the results as a causal influence.

Age is a categorical grouping variable in the primary model. An exploratory interaction model addresses the separate moderation question. No age cut-off is invented and no groups are merged after looking at results.

**Primary H0:** The coefficient for the awareness score equals zero, adjusting for age group. **H1:** It is nonzero. We use OLS with HC3 standard errors as an explicit equal-spacing approximation. A Spearman correlation is an ordinal sensitivity check; it does not adjust for age. The interaction jointly tests whether the slope varies across age groups. Small age groups limit this test.

Prepared with Codex assistance; Amanda should review and explain the analysis before submission.


In [ ]:
import statsmodels.formula.api as smf
from IPython.display import display, Markdown

agreement_scale = {'Strongly disagree': 1, 'Disagree': 2, 'Neutral': 3, 'Agree': 4, 'Strongly agree': 5}
sq2_columns = {
    'awareness': 'I am informed that my feed is shaped by personal information like my age, gender or location - transparency\xa0',
    'control': 'I feel I am in control of my feed',
}
sq2_all = pd.DataFrame(index=df.index)
for short, header in sq2_columns.items():
    answers = df[header]
    unexpected = set(answers.dropna()) - set(agreement_scale)
    if unexpected:
        raise ValueError(f'Unexpected response labels in {short}: {unexpected}')
    sq2_all[short] = answers.map(agreement_scale).astype(float)
sq2_all['age_group'] = df['What age group are you in?']
age_order = ['Under 18', '18-24', '25-34', '35-44', '45-54', '55-64', '65 and older']
unknown_ages = set(sq2_all.age_group.dropna()) - set(age_order)
if unknown_ages:
    raise ValueError(f'Unknown age bands: {unknown_ages}')
print('Missing values (before complete-case selection):')
display(sq2_all.isna().sum().to_frame('missing'))
sq2 = sq2_all.dropna().copy()
print('Included:', len(sq2), '| Excluded:', len(df) - len(sq2))
print('Repeated non-missing respondent IDs:', df['ID'].dropna().duplicated().sum())
# This check cannot establish independent sampling.
sq2_summary = sq2.groupby('age_group')[['awareness', 'control']].agg(['count', 'mean', 'median', 'std'])
display(sq2_summary.reindex(age_order).round(3))


## Response patterns
Bars show percentages within each age group. Empty age bands are omitted from the plot and remain visible in the table above. Age categories are not numeric ages.


In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ct = pd.crosstab(sq2['age_group'], sq2['control']).reindex(age_order).dropna(how='all')
ct = ct.reindex(columns=[1, 2, 3, 4, 5], fill_value=0)
ct.div(ct.sum(axis=1), axis=0).mul(100).plot.bar(stacked=True, ax=ax)
ax.set(xlabel='Age group', ylabel='Within-group percentage', title='SQ2: perceived control by age group')
ax.legend(title='Control score (1–5)', bbox_to_anchor=(1.02, 1))
plt.xticks(rotation=35)
plt.tight_layout()
plt.show()


## Primary adjusted association and ordinal sensitivity check
The coefficient is the expected difference in the control score per one-category increase in awareness, adjusting for age group. HC3 adjusts standard errors for unequal residual variances. It does not fix measurement error, dependence, omitted confounding or the equal-spacing assumption. A normality test on raw Likert scores is not a prerequisite for this analysis.


In [ ]:
sq2_model = smf.ols('control ~ awareness + C(age_group)', data=sq2).fit(cov_type='HC3')
X = sq2_model.model.exog
if np.linalg.matrix_rank(X) < X.shape[1]:
    raise ValueError('Design matrix is not full rank.')
sq2_results = pd.DataFrame({'coefficient': sq2_model.params, 'p': sq2_model.pvalues})
sq2_results[['CI_low', 'CI_high']] = sq2_model.conf_int()
display(sq2_results.round(5))
rho, rho_p = stats.spearmanr(sq2['awareness'], sq2['control'])
print(f'Unadjusted Spearman rho={rho:.4f}, p={rho_p:.6g}')
print(f'N={int(sq2_model.nobs)}, R-squared={sq2_model.rsquared:.4f}')
# Residual check: a discrete outcome naturally produces bands.
fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(sq2_model.fittedvalues, sq2_model.resid, alpha=.2)
ax.axhline(0, color='black')
ax.set(xlabel='Fitted control score', ylabel='Residual', title='SQ2: residual check')
plt.tight_layout()
plt.show()


## Exploratory moderation by age
A joint HC3 Wald test checks all awareness-by-age interaction terms together. A nonsignificant test is not evidence that every age-specific slope is identical. This analysis is exploratory, especially for the oldest and youngest groups.


In [ ]:
interaction_model = smf.ols('control ~ awareness * C(age_group)', data=sq2).fit(cov_type='HC3')
interaction_terms = [i for i, name in enumerate(interaction_model.params.index) if ':' in name]
R = np.eye(len(interaction_model.params))[interaction_terms]
interaction_test = interaction_model.wald_test(R, scalar=True)
interaction_p = float(interaction_test.pvalue)
print(f'Joint interaction Wald statistic={float(interaction_test.statistic):.4f}, df={len(interaction_terms)}, p={interaction_p:.6g}')
# Age-specific rank correlations are descriptive; no claims from multiple unadjusted tests.
rows = []
for age, group in sq2.groupby('age_group'):
    r = stats.spearmanr(group.awareness, group.control).statistic if group.awareness.nunique() > 1 and group.control.nunique() > 1 else np.nan
    rows.append({'age_group': age, 'n': len(group), 'Spearman_rho': r})
display(pd.DataFrame(rows).set_index('age_group').reindex(age_order).round(3))


## Reproducible conclusion
The text below is generated from the executed data rather than copied from an older analysis. The primary p-value is unadjusted; a conservative three-question Bonferroni value is also shown as a provisional group-level check. The group must agree its final testing family before reporting confirmatory results. This adjustment does not cover every exploratory analysis in the combined notebook.


In [ ]:
b = float(sq2_model.params['awareness'])
lo, hi = sq2_model.conf_int().loc['awareness']
p = float(sq2_model.pvalues['awareness'])
p3 = min(1., 3 * p)
evidence = 'evidence of an adjusted association' if p3 < .05 else 'insufficient evidence of an adjusted association after the provisional three-question correction'
moderation = 'evidence that slopes vary by age group' if interaction_p < .05 else 'insufficient evidence that slopes vary by age group'
display(Markdown(f"""Among **{len(sq2)} complete responses**, the awareness coefficient was **{b:.3f}** control-score points per awareness category (HC3 95% CI **{lo:.3f} to {hi:.3f}**; p = **{p:.6g}**; provisional Bonferroni p = **{p3:.6g}**). There is {evidence}. Model R² = **{sq2_model.rsquared:.3f}**.

The unadjusted ordinal sensitivity check gave Spearman rho = **{rho:.3f}** (p = **{rho_p:.6g}**). The exploratory interaction test gave p = **{interaction_p:.6g}**: {moderation}.

These results concern **awareness of feed shaping**, not the amount of personalisation. They do not establish that personalisation causes perceived control. Unequal and small age groups, single-item measures, complete-case selection, and observational sampling limit interpretation. No respondents are recorded in the 55–64 age band. Independence depends on recruitment and cannot be proven by checking duplicate IDs.
"""))
